# 01 · Bronze — Extracción y exploración inicial (EDA)
**Proyecto ETL 2026 · Grupo 6 · UAO** — ¿Cómo impactará la disminución de la natalidad y el envejecimiento poblacional
en la disponibilidad futura de la fuerza laboral en Corea del Sur?

Este notebook sigue la guía del *Laboratorio práctico de ETL* (partes 1 a 6) aplicada a nuestras fuentes:
extracción verificada, comprensión de cada dataset, perfil de calidad (nulos, únicos, duplicados, tipos),
estadísticos descriptivos y preguntas de negocio sobre los **datos originales**. Aquí **no se limpia ni se imputa nada**:
eso ocurre en Silver.

In [1]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
GOLD, SILVER, CTL = RAIZ/'data'/'gold', RAIZ/'data'/'silver', RAIZ/'data'/'ctl'

## 1. Extracción: qué quedó en Bronze
`data/bronze/_manifest.csv` registra cada versión ingerida (fuente, fecha, sha256, filas). Las descargas manuales de KOSIS se copian sin modificar; World Bank y OECD se descargan por API.

In [2]:
from src.ingestion.bronze import leer_vigente
man = pd.read_csv(RAIZ/'data'/'bronze'/'_manifest.csv')
vig = man[man.vigente]
print(f'{len(vig)} datasets vigentes · {vig.filas.sum():,} filas crudas')
vig[['fuente','dataset','tbl_id','fecha_extraccion','filas','columnas','rol']]

29 datasets vigentes · 21,425 filas crudas


,fuente,dataset,tbl_id,fecha_extraccion,filas,columnas,rol
0,KOSIS,censo_historico,DT_1IN0001,2026-10-02,2632,22,contexto
1,KOSIS,censo_registros,DT_1IN1502,2026-10-02,6580,13,complementaria
2,KOSIS,eaps_sexo_edad,DT_1DA7012S,2026-10-01,30,234,maestra
3,KOSIS,eaps_sido,DT_1DA7004S,2026-10-01,19,352,maestra
4,KOSIS,nacimientos_sexo,DT_1B8000K,2026-10-03,54,112,complementaria
5,KOSIS,poblacion_nacional,DT_1BPA001,2026-10-01,69,76,maestra
6,KOSIS,poblacion_sido,DT_1BPB001,2026-10-01,1242,60,maestra
7,KOSIS,proyeccion_escenarios,NaN,2026-10-01,1848,57,maestra
8,KOSIS,proyeccion_resumen,DT_1BPA002,2026-09-23,21,53,contraste
9,KOSIS,tfr_sido,DT_1B81A17,2026-10-01,18,209,maestra


In [3]:
log = pd.read_parquet(CTL/'log_cargas.parquet')
log[log.capa=='bronze'].groupby(['fuente','estado']).size().unstack(fill_value=0)

estado,duplicado,exito,omitido
fuente,,,
KOSIS,2,15,0
OECD,0,4,0
UNWPP,0,0,1
WB,0,10,0


Dos archivos llegaron **duplicados** (la misma tabla en `.xls` y `.xlsx`): Bronze los detecta por el hash del contenido tabular y no los ingiere dos veces. UN WPP queda **omitido** porque requiere un token (fuente de contraste opcional).

## 2. Verificación de la extracción (shape, columnas, head, muestra aleatoria)

In [4]:
ds_kosis = ['vitales_nacional','vitales_sigungu','tfr_sido','eaps_sido','eaps_sexo_edad','poblacion_nacional',
            'poblacion_sido','proyeccion_escenarios','censo_registros']
crudos = {d: leer_vigente('KOSIS', d)[0] for d in ds_kosis}
for d, df in crudos.items():
    print(f'{d:24s} shape={df.shape}  primeras columnas={list(df.columns[:4])}')

vitales_nacional         shape=(16, 57)  primeras columnas=['By items', '1970', '1971', '1972']
vitales_sigungu          shape=(19, 261)  primeras columnas=['By administrative divisions', '2000 | Live births (persons)', '2000 | Crude birth rate(per 1,000 population)', '2000 | Deaths (persons)']
tfr_sido                 shape=(18, 209)  primeras columnas=['By si(city), gun(county) and gu(borough)', '2000 | Total Fertility Rate', '2000 | Age-specific fertility rate:15-19', '2000 | 20-24 Years old']
eaps_sido                shape=(19, 352)  primeras columnas=['By province', '2025.08 | Population 15 years old and over (Thousand Person)', '2025.08 | Economically active population (Thousand Person)', '2025.08 | Employed persons (Thousand Person)']
eaps_sexo_edad           shape=(30, 234)  primeras columnas=['By gender', 'By age group', '2026.06 | Population 15 years old and over (Thousand Person)', '2026.06 | Economically active population (Thousand Person)']
poblacion_nacional       shape=(

In [5]:
df = crudos['vitales_nacional']
display(df.iloc[:5, :8]); display(df.sample(5, random_state=6).iloc[:, [0, -3, -2, -1]])

,By items,1970,1971,1972,1973,1974,1975,1976
0,Live births(persons),1006645,1024773,952780,965521,922823,874030,796331
1,Deaths(persons),258589,237528,210071,267460,248807,270657,266857
2,Natural increase(persons),748056,787245,742709,698061,674016,603373,529474
3,"Crude birth rate(per 1,000 population)",31.2,31.2,28.4,28.3,26.6,24.8,22.2
4,"Crude death rate(per 1,000 population)",8,7.2,6.3,7.8,7.2,7.7,7.4


,By items,2023,2024,2025
12,"Crude divorce rate(per 1,000 population)",1.8,1.8,1.7
14,Life expectancy at birth-male(age),80.6,80.8,-
6,Total fertility rate(persons),0.721,0.748,0.799
7,Masculinity of birth(persons),105.1,105,105.8
2,Natural increase(persons),-122483,-120252,-108627


In [6]:
df = crudos['eaps_sido']
display(df.iloc[:5, :5]); display(df.sample(5, random_state=6).iloc[:, :4])

,By province,2025.08 | Population 15 years old and over (Thousand Person),2025.08 | Economically active population (Thousand Person),2025.08 | Employed persons (Thousand Person),2025.08 | Unemployed persons (Thousand Person)
0,Total,45779,29559,28967,592
1,Seoul,8442,5374,5261,113
2,Jeonnam-Gwangju,2796,1830,1790,40
3,Gwangju,1275,805,787,18
4,Jeollanam-do,1520,1025,1004,22


,By province,2025.08 | Population 15 years old and over (Thousand Person),2025.08 | Economically active population (Thousand Person),2025.08 | Employed persons (Thousand Person)
5,Busan,2906,1731,1693
16,Gyeongsangbuk-do,2283,1543,1516
2,Jeonnam-Gwangju,2796,1830,1790
11,Gyeonggi-do,12221,7966,7783
8,Daejeon,1308,833,817


## 3. Comprensión inicial de cada dataset
¿Qué representa una fila? ¿Cuáles son los identificadores, las variables categóricas, numéricas y temporales? (Bronze guarda **todo como texto**, por diseño.)

In [7]:
def perfil(nombre, df, n_dims):
    periodos = sorted({c.split(' | ')[0] for c in df.columns[n_dims:] if not c.startswith('_col')})
    anuales = [p for p in periodos if p[:4].isdigit() and len(p.split()[0]) == 4]
    return {'dataset': nombre, 'filas': len(df), 'variables': df.shape[1],
            'identificadores (dimensiones)': ', '.join(df.columns[:n_dims]),
            'columnas de periodo/ítem': df.shape[1] - n_dims,
            'rango temporal': f"{min(anuales)[:4]}-{max(anuales)[:4]}" if anuales else '',
            'tipo en bronze': ', '.join(sorted({str(t) for t in df.dtypes}))}
dims = {'vitales_nacional':1,'vitales_sigungu':1,'tfr_sido':1,'eaps_sido':1,'eaps_sexo_edad':2,'poblacion_nacional':3,
        'poblacion_sido':6,'proyeccion_escenarios':5,'censo_registros':3}
pd.DataFrame([perfil(d, crudos[d], dims[d]) for d in ds_kosis])

,dataset,filas,variables,identificadores (dimensiones),columnas de periodo/ítem,rango temporal,tipo en bronze
0,vitales_nacional,16,57,By items,56,1970-2025,string
1,vitales_sigungu,19,261,By administrative divisions,260,2000-2025,string
2,tfr_sido,18,209,"By si(city), gun(county) and gu(borough)",208,2000-2025,string
3,eaps_sido,19,352,By province,351,2000-2025,string
4,eaps_sexo_edad,30,234,"By gender, By age group",232,2000-2025,string
5,poblacion_nacional,69,76,"가정별, 성별, 연령별",73,2000-2072,string
6,poblacion_sido,1242,60,"By scenarios, By province, By gender, By age, Item, UNIT",54,2000-2052,string
7,proyeccion_escenarios,1848,57,"시나리오별, 성별, 연령별, 항목, 단위",52,2022-2072,string
8,censo_registros,6580,13,"By administrative divisions(eup, myeon, dong), Item, UNIT",10,2016-2025,string


| Dataset | ¿Qué representa una fila (en bronze)? |
|---|---|
| vitales_nacional (DT_1B8000F) | Un ítem vital (nacimientos, TFR, esperanza de vida…) con un valor por año 1970-2025 en columnas |
| vitales_sigungu (DT_1B8000I) | Un si-do con 10 ítems × 26 años en columnas (encabezado doble periodo / ítem) |
| tfr_sido (DT_1B81A17) | Un si-do con la TFR y las tasas específicas por edad de la madre × 26 años |
| eaps_sido (DT_1DA7004S) | Un si-do con 9 indicadores laborales; mezcla columnas anuales y **mensuales** |
| eaps_sexo_edad (DT_1DA7012S) | Una combinación sexo × grupo de edad con 8 indicadores laborales por periodo |
| poblacion_nacional / poblacion_sido | Un escenario × (si-do) × sexo × grupo de edad con la población por año en columnas |
| proyeccion_escenarios | Uno de los 28 escenarios × sexo × edad con la población proyectada 2022-2072 |
| censo_registros (DT_1IN1502) | Una unidad administrativa (país, si-do, si-gun-gu, agregados urbanos) × ítem del censo |

**Variables categóricas:** territorio, sexo, grupo de edad, escenario, ítem. **Numéricas:** los valores de cada
celda año-ítem (llegan como texto). **Temporales:** los encabezados de periodo (`2025`, `2025 Year`, `2022 년`, `2026.05 p)`).

## 4. Perfil inicial de calidad
### 4.1 Valores faltantes
KOSIS marca la ausencia de dato con `-` o con celdas vacías; pandas no los ve como nulos porque Bronze guarda texto. Se cuantifican aquí, **sin eliminar ni imputar**.

In [8]:
def faltantes(nombre, df, n_dims):
    v = df.iloc[:, n_dims:]
    v = v.loc[:, ~v.columns.str.startswith('_col')]
    marcados = v.isin(['-', '', 'x', '...']).sum().sum() + v.isna().sum().sum()
    return {'dataset': nombre, 'celdas_valor': v.size, 'faltantes': int(marcados),
            'pct_faltantes': round(marcados / v.size * 100, 2)}
nul = pd.DataFrame([faltantes(d, crudos[d], dims[d]) for d in ds_kosis]).sort_values('pct_faltantes', ascending=False)
nul

,dataset,celdas_valor,faltantes,pct_faltantes
1,vitales_sigungu,4940,328,6.64
3,eaps_sido,6669,378,5.67
0,vitales_nacional,896,34,3.79
2,tfr_sido,3744,96,2.56
8,censo_registros,65800,880,1.34
6,poblacion_sido,65826,828,1.26
4,eaps_sexo_edad,6960,0,0.00
5,poblacion_nacional,5037,0,0.00
7,proyeccion_escenarios,94248,0,0.00


In [9]:
s = pd.read_parquet(SILVER/'sin_dato.parquet')
s.groupby(['dataset','motivo']).size().rename('celdas').reset_index().sort_values('celdas', ascending=False)

,dataset,motivo,celdas
1,censo_historico,territorio aún no existía como si-do,4818
0,censo_historico,grupo de edad o territorio no publicado en ese censo,3675
5,poblacion_sido,territorio aún no existía como si-do,792
10,vitales_sido,territorio aún no existía como si-do,302
3,eaps_sido,región combinada (Jeonnam-Gwangju) sólo publicada en 2025,225
2,eaps_sido,EAPS publica Sejong desde 2017,153
13,vitales_sigungu,territorio aún no existía como si-do,120
6,tfr_sido,territorio aún no existía (Sejong < 2012),84
11,vitales_sido_mensual,región combinada (Jeonnam-Gwangju) sólo publicada en 2025,75
4,nacimientos_sexo,territorio aún no existía como si-do,72


**Interpretación de los nulos.** La mayoría son *esperados* por el funcionamiento de la fuente: Sejong no existía
como si-do antes de 2012 (y la EAPS lo publica desde 2017), la tabla de vida 2025 aún no se publicaba en la fecha de descarga,
la mortalidad infantil anual se publica desde 2009 y los censos antiguos no traen todos los grupos de edad.
Los únicos faltantes que representan un problema de calidad son los de la región combinada **Jeonnam-Gwangju**
(sólo existe en 2025) — en Silver se rechaza con motivo trazado.

### 4.2 Valores únicos (cardinalidad) y 4.3 duplicados

In [10]:
filas = []
for d in ds_kosis:
    df = crudos[d]
    for c in df.columns[:dims[d]]:
        filas.append({'dataset': d, 'variable': c, 'n_unicos': df[c].nunique(),
                      'ejemplos': ', '.join(df[c].astype(str).unique()[:4])})
card = pd.DataFrame(filas); card

,dataset,variable,n_unicos,ejemplos
0,vitales_nacional,By items,16,"Live births(persons), Deaths(persons), Natural increase(persons), Crude birt..."
1,vitales_sigungu,By administrative divisions,19,"Whole country, Seoul, Busan, Daegu"
2,tfr_sido,"By si(city), gun(county) and gu(borough)",18,"Whole country, Seoul, Busan, Daegu"
3,eaps_sido,By province,19,"Total, Seoul, Jeonnam-Gwangju, Gwangju"
4,eaps_sexo_edad,By gender,3,"Total, Male, Female"
5,eaps_sexo_edad,By age group,10,"Total, 15-29 Years old, 15-19 Years old, 20-29 Years old"
6,poblacion_nacional,가정별,1,중위 추계(기본 추계: 출산율-중위 / 기대수명-중위 / 국제순이동-중위)
7,poblacion_nacional,성별,3,"전체, 남자, 여자"
8,poblacion_nacional,연령별,23,"계, 0 - 4세, 5 - 9세, 10 - 14세"
9,poblacion_sido,By scenarios,1,Medium


In [11]:
dup = pd.DataFrame([{'dataset': d, 'filas_duplicadas_completas': int(crudos[d].duplicated().sum()),
                     'claves_dimension_repetidas': int(crudos[d].duplicated(subset=list(crudos[d].columns[:dims[d]])).sum())}
                    for d in ds_kosis]); dup

,dataset,filas_duplicadas_completas,claves_dimension_repetidas
0,vitales_nacional,0,0
1,vitales_sigungu,0,0
2,tfr_sido,0,0
3,eaps_sido,0,0
4,eaps_sexo_edad,0,0
5,poblacion_nacional,0,0
6,poblacion_sido,0,0
7,proyeccion_escenarios,0,0
8,censo_registros,20,1400


In [12]:
c = crudos['censo_registros']; t = c.iloc[:, 0].str.strip()
rep = t[c.Item.eq('Total population[Person]')].value_counts(); rep[rep > 1]

By administrative divisions(eup, myeon, dong)
Total for dongs     15
Total for eups      15
Total for myeons    15
Jung-gu              6
Dong-gu              6
Nam-gu               6
Seo-gu               5
Buk-gu               5
Gangseo-gu           2
Gunwi-gun            2
Sejong-si            2
Goseong-gun          2
Chungju-si           2
Name: count, dtype: int64[pyarrow]

En el censo de registros se repiten nombres de distritos (`Jung-gu` existe en varias ciudades) y **Sejong** aparece
dos veces (como si-do y como ciudad). Como sólo se usan el nivel nacional y los si-do, el único duplicado relevante es
Sejong: Silver conserva la primera aparición y traza las 70 filas rechazadas.
Las proyecciones por escenario repiten claves por diseño (28 escenarios): la clave incluye el escenario.

### 4.4 Tipos de datos
Todo llega como texto (`string`). Columnas a revisar antes de analizar: valores con `-`, encabezados de periodo heterogéneos, unidades en **miles** en la EAPS, y la codificación CP949 / EUC-KR de dos archivos.

## 5. Estadísticos descriptivos (conversión numérica sólo para describir)

In [13]:
def a_largo_num(df, n_dims):
    l = df.melt(id_vars=list(df.columns[:n_dims]), var_name='col', value_name='v')
    l['valor'] = pd.to_numeric(l.v.astype(str).str.replace(',', ''), errors='coerce')
    l['periodo'] = l.col.str.split(' | ', regex=False).str[0]; l['item'] = l.col.str.split(' | ', regex=False).str[1]
    return l
v = a_largo_num(crudos['vitales_nacional'], 1)
v.groupby('By items').valor.describe().round(2)

,count,mean,std,min,25%,50%,75%,max
By items,,,,,,,,
"Crude birth rate(per 1,000 population)",56.0,14.35,7.17,4.50,9.00,14.10,17.35,31.20
"Crude death rate(per 1,000 population)",56.0,5.91,0.86,5.00,5.20,5.60,6.40,8.00
"Crude divorce rate(per 1,000 population)",56.0,1.60,0.85,0.30,0.85,1.75,2.30,3.40
"Crude marriage rate(per 1,000 population)",56.0,7.46,1.92,3.70,6.30,7.40,9.33,10.60
Deaths(persons),56.0,261605.54,34152.61,210071.00,242396.75,247233.00,267518.00,372939.00
Divorces(cases),56.0,76007.55,44731.98,11361.00,33966.25,89640.50,115346.50,166617.00
"Infant mortality rate(per 1,000 live births)",25.0,3.25,0.94,2.00,2.70,3.00,3.40,5.40
Life expectancy at birth-female(age),55.0,77.84,6.54,65.80,72.25,78.70,83.80,86.60
Life expectancy at birth-male(age),55.0,70.46,7.29,58.70,63.70,70.70,77.05,80.80


In [14]:
e = a_largo_num(crudos['eaps_sido'], 1)
e = e[e.periodo.str.fullmatch(r'\d{4}')]
e.groupby('item').valor.describe().round(1)

,count,mean,std,min,25%,50%,75%,max
item,,,,,,,,
Economically active population (Thousand Person),452.0,2996.9,5935.2,132.0,752.5,1046.5,1703.5,29599.0
Economically inactive population (Thousand Person),452.0,1794.8,3538.1,79.0,484.0,630.0,1040.8,16773.0
Employed persons (Thousand Person),452.0,2893.2,5729.4,129.0,729.8,1023.0,1634.8,28769.0
Employment to population ratio (%),452.0,60.6,3.1,53.4,58.4,60.1,62.4,70.9
Employment to population ratio(15-64) (%),452.0,65.3,3.4,55.9,62.8,65.2,67.7,76.0
Labor Force Participation rate (%),452.0,62.5,3.0,56.3,60.2,62.4,64.3,72.3
Population 15 years old and over (Thousand Person),452.0,4791.6,9469.9,211.0,1253.8,1650.0,2816.5,45762.0
Unemployed persons (Thousand Person),452.0,103.8,208.3,3.0,22.0,31.0,62.2,1108.0
Unemployment rate (%),452.0,3.2,0.9,1.3,2.5,3.1,3.8,7.1


## 6. Preguntas de negocio sobre los datos originales (pandas)

In [15]:
nac = v[v['By items'].eq('Live births(persons)')].set_index('periodo').valor
tfr = v[v['By items'].eq('Total fertility rate(persons)')].set_index('periodo').valor
ev = v[v['By items'].eq('Life expectancy at birth-total(age)')].set_index('periodo').valor.dropna()
print('P1. Año con más nacimientos desde 1970:', nac.idxmax(), f'({nac.max():,.0f})')
print('P2. Año con la TFR más baja:', tfr.idxmin(), f'({tfr.min()})')
print('P3. Primer año con más defunciones que nacimientos:',
      v[v['By items'].eq('Natural increase(persons)')].set_index('periodo').valor.lt(0).idxmax())
print('P4. Ganancia de esperanza de vida', ev.index[0], '→', ev.index[-1], ':', round(ev.iloc[-1] - ev.iloc[0], 1), 'años')
print('P5. Caída de nacimientos 2000 → 2025:', f"{(nac['2025'] / nac['2000'] - 1) * 100:.1f} %")

P1. Año con más nacimientos desde 1970: 1971 (1,024,773)
P2. Año con la TFR más baja: 2023 (0.721)
P3. Primer año con más defunciones que nacimientos: 2020
P4. Ganancia de esperanza de vida 1970 → 2024 : 21.4 años
P5. Caída de nacimientos 2000 → 2025: -60.3 %


In [16]:
t = a_largo_num(crudos['tfr_sido'], 1)
t = t[t.item.eq('Total Fertility Rate') & t.periodo.eq('2025')]
t = t.set_index(t.iloc[:, 0].str.strip()).valor
print('P6. Si-do con la TFR más baja en 2025:', t.drop('Whole country').idxmin(), t.drop('Whole country').min())
print('P7. Si-do con la TFR más alta en 2025:', t.drop('Whole country').idxmax(), t.drop('Whole country').max())
ea = e[e.item.eq('Labor Force Participation rate (%)') & e.periodo.eq('2025')].set_index('By province').valor
print('P8. Si-do con mayor participación laboral en 2025:', ea.drop('Total').idxmax(), ea.drop('Total').max())
es = a_largo_num(crudos['eaps_sexo_edad'], 2)
f60 = es[(es['By gender']=='Female') & es['By age group'].str.startswith('60') & es.item.eq('Labor Force Participation rate (%)')
         & es.periodo.isin(['2000','2025'])].set_index('periodo').valor
print('P9. Participación de mujeres 60+: 2000 =', f60['2000'], '% → 2025 =', f60['2025'], '%')
c = crudos['censo_registros']; fx = c[(c.iloc[:,0].str.strip()=='Whole country') & c.Item.str.startswith('Foreigner-Total')]
print('P10. Población extranjera residente 2016 → 2025:', fx['2016 Year'].iloc[0], '→', fx['2025 Year'].iloc[0])

P6. Si-do con la TFR más baja en 2025: Seoul 0.632
P7. Si-do con la TFR más alta en 2025: Jeollanam-do 1.094
P8. Si-do con mayor participación laboral en 2025: Jeju-do 71.4


P9. Participación de mujeres 60+: 2000 = 30.1 % → 2025 = 40.2 %
P10. Población extranjera residente 2016 → 2025: 1413758 → 2109057


## Conclusiones de la exploración (hallazgos)
1. **Formatos heterogéneos:** CSV UTF-8, CSV **CP949**, XLSX y "XLS" que en realidad es XML Spreadsheet 2003 en **EUC-KR**
   con XML mal formado → se requirieron 4 lectores específicos.
2. **Duplicados de archivo:** dos tablas se descargaron en `.xls` y `.xlsx` con contenido idéntico (detectado por hash).
3. **Huecos de la fuente:** la descarga DT_1B8000H no contiene el año **2022** ni la TFR de varios años; DT_1B8000I
   (mismo productor) sí está completa → se elige como fuente maestra regional.
4. **Periodicidades mezcladas:** EAPS y vitales mensuales traen columnas mensuales (`2026.05 p)`) junto con las anuales.
5. **Cambios territoriales:** Sejong (2012), región combinada Jeonnam-Gwangju (sólo 2025), nombres nuevos
   `Gangwon-State` / `Jeonbuk-State`, ciudades metropolitanas creadas entre 1963 y 1997.
6. **Unidades:** la EAPS publica niveles en **miles de personas**; las tasas a 1 decimal.
7. **Agregados redundantes:** `80 Years old & over` coexiste con 80-84, 85-89…; sumarlo duplicaría población.
8. **Proyección disfrazada de historia:** las tablas de población 2000-2072 son de proyección: 2023-2025 **no son observados**.
9. **Faltantes mayormente esperados** (Sejong, tabla de vida 2025, series que empiezan después): no se imputan.
10. Los datos confirman el problema: nacimientos −60 % entre 2000 y 2025, TFR mínima de 0,72 en 2023 y crecimiento
    natural negativo desde 2020.